<a href="https://colab.research.google.com/github/DarrenDsa6/AI-Recruiter-Intelligence-Assistant/blob/main/TinyLlama_Text2SQL_LoRA_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Environment Setup

In [ ]:
# Install required libraries
!pip install -q \
"transformers==5.0.0" \
"peft==0.18.1" \
"accelerate==1.13.0" \
"datasets==4.8.4" \
"trl==1.1.0" \
"sentencepiece==0.2.1" \
"protobuf==5.29.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.6 MB/s eta 0:00:00


In [ ]:
import sys
import torch

print("Python: ", sys.version)
print("Pytorch: ", torch.__version__)

# Check GPU availability
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version: ", torch.version.cuda)
    print("GPU name: ", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory/1024**3:.2f} GB")

Python:  3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Pytorch:  2.10.0+cu128
CUDA available:  True
CUDA version:  12.8
GPU name:  Tesla T4
GPU memory: 14.56 GB


#2. Load the Base Model in FullPrecision16(FP16)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Starting Base Model Loading")

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID) # Loads the particular type of tokenizer based on the MODEL_ID
if tokenizer.pad_token_id is None:
  print("Model has no padding token. Adding padding token")
  tokenizer.pad_token_id = tokenizer.eos_token_id # EOS = End of Sentence token
  # tokens1 -> ["I", "Love", "Chat", "GPT", "<pad>", "<pad>"] == Right means add padding token to right
  # tokens2 -> ["ML", "is", "a", "Branch", "of", "AI"]
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto") #Loads the model itself
print("Loading of Base Model Complete")

model.config.use_cache = False #Required for gradient checkpointing
model.config.pretraining_tp = 1 #This tells that the model is in one GPU and is not distributed(TP = tensor parallelism)

n_params = sum(p.numel() for  p in model.parameters())
print(f"Number of parameters: {n_params/1e6:.1f}M")
print(f"Memory footprint: {model.get_memory_footprint()/1024**3:.2f}GB")

Starting Base Model Loading


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading of Base Model Complete
Number of parameters: 1100.0M
Memory footprint: 2.05GB


# 3. Define a Prompt and Test the Base Model
### We use TinyLlama's chat template.
#### Task = Text to SQL(Given a table schema and a question, produces a query)

In [ ]:
def build_prompt(schema: str, question: str) -> str:
  system = (
    "You are an SQL generator. Input: schema + question. Output: ONLY the SQL query in a code block."
    "Do not add explanations, comments, or extra tables. Use only the schema provided."
    "If the schema does not support the question, return an error message: Schema does not support query."
    )
  user = f"Schema: {schema}\n\nQuestion: {question}"
  messages = [
      {"role": "system", "content": system},
      {"role": "user", "content": user}
  ]
  return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:
schema = "CREATE TABLE employees (id INT PRIMARY KEY, name TEXT NOT NULL, department_id INT, salary BIGINT, hire_date DATE, manager_id INT)"

question = "List the names of all the employees and show their salary"

prompt = build_prompt(schema, question)
print(prompt)


<|system|>
You are an SQL generator. Input: schema + question. Output: ONLY the SQL query in a code block.Do not add explanations, comments, or extra tables. Use only the schema provided.If the schema does not support the question, return an error message: Schema does not support query.</s>
<|user|>
Schema: CREATE TABLE employees (id INT PRIMARY KEY, name TEXT NOT NULL, department_id INT, salary BIGINT, hire_date DATE, manager_id INT)

Question: List the names of all the employees and show their salary</s>
<|assistant|>



In [ ]:
@torch.no_grad() #Means dont calculate the gradients
def generate_results(prompt: str, max_new_tokens: int = 200) -> str:
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device) #Here input is converted into tokenId.
  #First sentence is converted to tokens and then tokens are converted to tokenIds
  print("Prompt:", prompt)
  #Here the model is called with the input prompt. The output is also as tokens
  outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                           do_sample=False,
                           pad_token_id=tokenizer.pad_token_id,
                           eos_token_id=tokenizer.eos_token_id) # Corrected argument name to eos_token_id
  input_length = inputs["input_ids"].shape[1]
  new_tokens = outputs[0][input_length:] # Changed output_ids to outputs and input to input_length
  #Return the output after decoding the tokens and converting it into response
  return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
response = generate_results(prompt=prompt)
print("Response", response)

Prompt: <|system|>
You are an SQL generator. Input: schema + question. Output: ONLY the SQL query in a code block.Do not add explanations, comments, or extra tables. Use only the schema provided.If the schema does not support the question, return an error message: Schema does not support query.</s>
<|user|>
Schema: CREATE TABLE employees (id INT PRIMARY KEY, name TEXT NOT NULL, department_id INT, salary BIGINT, hire_date DATE, manager_id INT)

Question: List the names of all the employees and show their salary</s>
<|assistant|>

Response Here's a possible SQL query that meets the given requirements:

```sql
SELECT e.name, e.salary
FROM employees e
JOIN departments d ON e.department_id = d.id
WHERE d.id = (SELECT m.id
              FROM managers m
              WHERE m.employee_id = e.id
              ORDER BY m.manager_id DESC
              LIMIT 1);
```

This query selects all employees from the `employees` table, joins it to the `departments` table using the `id` column of the `depar

# 4. Load and Prepare the dataset



In [ ]:
from datasets import load_dataset

raw = load_dataset("b-mc2/sql-create-context", split="train")
print("Full Dataset Size", len(raw))
print("Example rows", raw[:5])

README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Full Dataset Size 78577
Example rows {'answer': ['SELECT COUNT(*) FROM head WHERE age > 56', 'SELECT name, born_state, age FROM head ORDER BY age', 'SELECT creation, name, budget_in_billions FROM department', 'SELECT MAX(budget_in_billions), MIN(budget_in_billions) FROM department', 'SELECT AVG(num_employees) FROM department WHERE ranking BETWEEN 10 AND 15'], 'question': ['How many heads of the departments are older than 56 ?', 'List the name, born state and age of the heads of departments ordered by age.', 'List the creation year, name and budget of each department.', 'What are the maximum and minimum budget of the departments?', 'What is the average number of employees of the departments whose rank is between 10 and 15?'], 'context': ['CREATE TABLE head (age INTEGER)', 'CREATE TABLE head (name VARCHAR, born_state VARCHAR, age VARCHAR)', 'CREATE TABLE department (creation VARCHAR, name VARCHAR, budget_in_billions VARCHAR)', 'CREATE TABLE department (budget_in_billions INTEGER)', 'CREA

In [ ]:
#The dataset has a lot of data and we do not need so much for finetuning
#Hence we will keep it small
raw = raw.shuffle(seed=42).select(range(3500))
split = raw.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("Full Dataset Size", len(raw))
print("Example rows", raw)
print("Training Dataset Length", len(train_ds))
print("Evaluation Dataset Length", len(eval_ds))

Full Dataset Size 3500
Example rows Dataset({
    features: ['answer', 'question', 'context'],
    num_rows: 3500
})
Training Dataset Length 3150
Evaluation Dataset Length 350


In [ ]:
train_ds[0]

{'answer': 'SELECT COUNT(kickoff_)[a_] FROM table_11406866_2 WHERE "week" = "week"',
 'question': 'what is the total number of\xa0kickoff [a ]\xa0where\xa0week\xa0is week',
 'context': 'CREATE TABLE table_11406866_2 (a_ VARCHAR, kickoff_ VARCHAR)'}

In [ ]:
def format_example(row):
  system = (
      "You are an SQL assistant. Given a table schema and question, "
      "reply with ONLY SQL Query, Nothing else."
  )
  user = f"Schema: {row['context']}\n\nQuestion: {row['question']}"
  assistant = row["answer"]
  messages = [
      {"role": "system", "content": system},
      {"role": "user", "content": user},
      {"role": "assistant", "content": assistant}
  ]
  text = tokenizer.apply_chat_template(messages, tokenize=False)
  return {"text": text}

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(format_example, remove_columns=eval_ds.column_names)

print("\n---Formatted Training Example---\n")
print(train_ds[0]["text"][:1000])
train_ds[0].keys()

Map:   0%|          | 0/3150 [00:00<?, ? examples/s]

Map:   0%|          | 0/350 [00:00<?, ? examples/s]


---Formatted Training Example---

<|system|>
You are an SQL assistant. Given a table schema and question, reply with ONLY SQL Query, Nothing else.</s>
<|user|>
Schema: CREATE TABLE table_11406866_2 (a_ VARCHAR, kickoff_ VARCHAR)

Question: what is the total number of kickoff [a ] where week is week</s>
<|assistant|>
SELECT COUNT(kickoff_)[a_] FROM table_11406866_2 WHERE "week" = "week"</s>



dict_keys(['text'])

In [ ]:
print("Formatted Text:", train_ds[0]["text"])
print("="*70)

Formatted Text: <|system|>
You are an SQL assistant. Given a table schema and question, reply with ONLY SQL Query, Nothing else.</s>
<|user|>
Schema: CREATE TABLE table_11406866_2 (a_ VARCHAR, kickoff_ VARCHAR)

Question: what is the total number of kickoff [a ] where week is week</s>
<|assistant|>
SELECT COUNT(kickoff_)[a_] FROM table_11406866_2 WHERE "week" = "week"</s>



#5. Attach LoRA(Low Rank Adaptation) Adaptors

In [ ]:
from peft import LoraConfig, get_peft_model

#Enable gradient checkpointing to save VRAM which backward pass
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model = get_peft_model(model, lora_config)

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


#6. Train the Model on the DataSet(FineTune)

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./tinyllama-sql-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=10,
    optim="adamw_torch",
    logging_steps=10,
    max_grad_norm=0.3,
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none"
)

#Pre tokenize so we dont have to depent on SFTTrainers text-field Handling
def tokenize(batch):
  out = tokenizer(
      batch["text"],
      truncation=True,
      max_length=512,
      padding=False
  )
  return out

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

trainer = SFTTrainer(
    model=model,
    args=sft_config, # Pass the SFTConfig object here
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Map:   0%|          | 0/3150 [00:00<?, ? examples/s]

Map:   0%|          | 0/350 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
50,0.637564,0.606933
100,0.589041,0.568583
150,0.601532,0.554475


TrainOutput(global_step=197, training_loss=0.7061262288069362, metrics={'train_runtime': 494.3361, 'train_samples_per_second': 6.372, 'train_steps_per_second': 0.399, 'total_flos': 2673721602514944.0, 'train_loss': 0.7061262288069362})

#7. Compare Model Post Fine Tuning

In [ ]:
PROBES = [
    {
        "schema": "CREATE TABLE products (product_id INT PRIMARY KEY, product_name TEXT, price DECIMAL)",
        "question": "List all product names and their prices."
    },
    {
        "schema": "CREATE TABLE orders (order_id INT PRIMARY KEY, customer_id INT, order_date DATE, total_amount DECIMAL)",
        "question": "What is the total amount of all orders made by customer with ID 101?"
    },
    {
        "schema": "CREATE TABLE users (user_id INT PRIMARY KEY, username TEXT)",
        "question": "Show the average salary of all users."
    }
]


model.config.use_cache = True
model.eval()

print("="*70)
print("Fine Tuned Model(After LoRA)")
print("="*70)

probe_generated_sqls = []
for i, probe in enumerate(PROBES):
    print(f"--- Probe {i+1} ---")
    print(f"Schema: {probe['schema']}")
    print(f"Question: {probe['question']}")
    prompt = build_prompt(probe["schema"], probe["question"])
    response = generate_results(prompt=prompt)
    print("Response:", response)
    probe_generated_sqls.append(response)
    print("-" * 20)

print("All generated SQLs from fine-tuned model:", probe_generated_sqls)


Fine Tuned Model(After LoRA)
--- Probe 1 ---
Schema: CREATE TABLE products (product_id INT PRIMARY KEY, product_name TEXT, price DECIMAL)
Question: List all product names and their prices.
Prompt: <|system|>
You are an SQL generator. Input: schema + question. Output: ONLY the SQL query in a code block.Do not add explanations, comments, or extra tables. Use only the schema provided.If the schema does not support the question, return an error message: Schema does not support query.</s>
<|user|>
Schema: CREATE TABLE products (product_id INT PRIMARY KEY, product_name TEXT, price DECIMAL)

Question: List all product names and their prices.</s>
<|assistant|>

Response: SELECT product_name, price FROM products ORDER BY product_id
--------------------
--- Probe 2 ---
Schema: CREATE TABLE orders (order_id INT PRIMARY KEY, customer_id INT, order_date DATE, total_amount DECIMAL)
Question: What is the total amount of all orders made by customer with ID 101?
Prompt: <|system|>
You are an SQL genera